In [16]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

# 1. 문서 로더

In [17]:
!pip install pypdf

In [22]:
from langchain_community.document_loaders import PyPDFLoader

In [23]:
loader = PyPDFLoader("data/Samsung_Card_Manual_Korean_1.3.pdf")
pages = loader.load()  # List[Document] 형태로 반환

In [24]:
len(pages)

6

In [25]:
pages[0].page_content[:100]
pages[0].metadata

{'producer': 'Microsoft® Word 2010',
 'creator': 'Microsoft® Word 2010',
 'creationdate': '2019-01-02T09:40:06+09:00',
 'author': 'master',
 'moddate': '2019-01-02T09:40:06+09:00',
 'source': 'data/Samsung_Card_Manual_Korean_1.3.pdf',
 'total_pages': 6,
 'page': 0,
 'page_label': '1'}

# 2. 텍스트 스플리터

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.split_documents(pages)

In [34]:
len(docs)

9

In [28]:
docs[0].page_content

'사용설명서           SAMSUNG PROPRIETARY \nRevision1.3 \n1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n. \n \n법적 고지 사항 \n삼성 전자는 통지 없이 제품, 정보 및 사양을 변경할 권리를 보유합니다.  \n여기에 언급된 제품 및 사양은 참조용으로만 사용되며, 여기에 언급된 모든 정보는 공지 없이 변경될 수 있고 \n어떠한 종류의 보증도 없이 "있는 그대로" 제공됩니다. 본 문서 및 여기에 명시된 모든 정보는 삼성 전자의 \n유일하고 배타적인 자산으로 보유됩니다. 본 문서에 따라 어떠한 특허권, 저작권, 마스크워크, 상표 또는 기타 \n지적 재산권도 묵시적, 금반언적 또는 기타 어떤 방식으로도 한 당사자가 다른 당사자에게 부여할 수 없습니다. \n삼성 제품은 생명 보조기구, 구명의료기, 의료 기기, 안전 장비 또는 제품의 오류로 인해 사망, 부상 또는 물리적'

In [29]:
docs[1].page_content

'삼성 제품은 생명 보조기구, 구명의료기, 의료 기기, 안전 장비 또는 제품의 오류로 인해 사망, 부상 또는 물리적 \n상해를 야기할 수 있는 유사 제품, 군용 또는 방어 기기 또는 특정 계약/조항이 적용될 수 있는 정부의 조달에 \n사용할 의도로 설계되지 않았습니다. 삼성 제품에 관한 업데이트 또는 추가 정보는 가까운 삼성 대리점에 \n문의하십시오. 모든 브랜드 이름, 상표 및 등록 상표는 해당 소유권자가 소유합니다.  \nCopyright, 2019 Samsung Electronics Co., Ltd. All rights reserved.  \n \nCOPYRIGHT 2019  \n본 자료의 저작권은 삼성 전자에 있습니다. 본 자료의 전부 또는 일부를 무단으로 복제하거나 사용 또는 공개하는 \n행위는 엄격히 금지되며 저작권법에 위배됩니다. \n상표 및 서비스 표시'

In [30]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [31]:
embeddings = OpenAIEmbeddings()

# 3. 임베딩

In [32]:
!pip install faiss-cpu --no-cache-dir

In [33]:
from langchain_community.vectorstores import FAISS

# 4. 벡터 저장

In [35]:
vectordb = FAISS.from_documents(docs, embeddings)

In [36]:
vectordb

# 5. 검색

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})  # 검색기 생성
# k는 반환할 청크 수입니다. 도메인과 청크 크기에 따라 조정합니다.

# 6. 프롬프트 구성

In [38]:
from langchain_core.prompts import ChatPromptTemplate

In [39]:
msg = """
너는 삼성전자 메모리카드 매뉴얼에 대한 전문 어시스턴트이다.
다음의 참고 문서를 바탕으로 질문에 정확하게 답하라.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하라."""

In [40]:
prompt = ChatPromptTemplate.from_template(msg)

# 7. 응답 생성

In [41]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [42]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "이 유틸리티는 동시에 몇 개의 메모리카드나 UFD를 인식할 수 있나?"  # 예시 질의
answer = rag_chain.invoke(query)

In [43]:
answer

'이 유틸리티는 동시에 최대 8개의 메모리 카드나 UFD를 인식할 수 있습니다.'

In [ ]:
llm.invoke(query) # retriever(검색) rag 결과 vs. 단순 llm 결과 차이 확인

AIMessage(content='유틸리티가 동시에 인식할 수 있는 메모리 카드나 USB 플래시 드라이브(UFD)의 개수는 사용 중인 하드웨어와 소프트웨어에 따라 다릅니다. 일반적으로, 대부분의 운영 체제는 여러 개의 USB 장치를 동시에 인식할 수 있으며, USB 허브를 사용하면 더 많은 장치를 연결할 수 있습니다. 그러나 각 장치의 성능이나 전원 공급, 드라이버 호환성 등에 따라 인식 가능 개수가 제한될 수 있습니다.\n\n정확한 개수를 알고 싶다면, 사용 중인 유틸리티의 문서나 지원 페이지를 참조하거나, 해당 유틸리티의 개발자에게 문의하는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 34, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a80228f72e', 'id': 'chatcmpl-DU1l0BoAdgmGwVjltHnLfxx7e1jRt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d84c4-8996-70a2-b3a8-3110be5146c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'i

# 질문 3개, llm vs. rag_chain 비교